In [0]:
# ingestion notebook

# Spark
from pyspark.sql.functions import *
from pyspark.sql import DataFrame
from pyspark.sql.types import *
from delta.tables import *

# API
import requests
import json
from datetime import datetime
from typing import Dict, Any, List, Optional

# Logging
import logging

In [0]:
# basic logging - werd aangeraden door Copilot

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


In [0]:
# dynamic API parameters -- dictionary van parameters die meegegeven kunnen worden aan de functie

API_CONFIG = {
    "base_url": "https://meowfacts.herokuapp.com/",
    "headers": None,
    "params": None,
    "pagination": False,
    "data_key": None,
    "catalog": "raw",
    "schema": "api",
    "table": "cat_facts"
}

In [0]:
# function API loader - functie voor de api

def get_api_data(config: dict):  
    # ophalen van juiste key-value pairs
    url = config.get("base_url")
    headers = config.get("headers")
    params = config.get("params")
    pagination = config.get("pagination")
    data_key = config.get("data_key")

    logger.info(f"Calling API: {url}")

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    try:
        result = response.json()
    except Exception as e: # als het resultaat niet in json formaat weergegeven kan worden
        return {"error": f"JSON decode failed: {str(e)}", "status_code": response.status_code, "text": response.text}

    if response.status_code != 200: # als niet succesvol
        return {"error": response.status_code}
    else:
        return result
    
# get_api_data(API_CONFIG)

In [0]:
def load():
    raw_response = []

    response_data = get_api_data(API_CONFIG)

    raw_response.append({
        'api_response': json.dumps(response_data),  # Store entire response as JSON string
        'load_timestamp': datetime.now()
    })

    if raw_response:
    # Define schema for raw storage
        schema = StructType([
            StructField('api_response', StringType(), False),
            StructField('load_timestamp', TimestampType(), False)
        ])

        df = spark.createDataFrame(raw_response, schema=schema)

        print(f" \nRunning query CREATE SCHEMA IF NOT EXISTS {API_CONFIG['catalog']}.{API_CONFIG['schema']}")
        query = f"""
            CREATE SCHEMA IF NOT EXISTS {API_CONFIG['catalog']}.{API_CONFIG['schema']}
        """
        spark.sql(query)

        df.write.format("delta").mode("append").saveAsTable(f"{API_CONFIG['catalog']}.{API_CONFIG['schema']}.{API_CONFIG['table']}")
        print(f"💾 Loaded {len(raw_response)} raw API responses to {API_CONFIG['catalog']}.{API_CONFIG['schema']}.{API_CONFIG['table'] }")
                           
    else:
        print(f" No API response received")


In [0]:
# convert json API data to key-value records

def json_to_df(api_data: dict) -> list:

    if not api_data:
        raise ValueError("No data received from API")

    records = [] # maak een lege lijst aan waar de records in kunnen worden toegevoegd
    
    for key, value in api_data.items(): 
        if isinstance(value, list): # alleen dictionaries die een lijst bevatten als value
            for item in value:
                records.append({ # voeg voor elk item een dictionary toe aan de list records
                    "key": key,
                    "value": str(item)
                })
        elif isinstance(value, dict):
            records.append({
                "key": key,
                "value": json.dumps(value)
            })
        else:
            records.append({
                "key": key,
                "value": str(value)
            })
    return spark.createDataFrame(records) # maak van API data een Spark DataFrame



    # return spark.createDataFrame(api_data)


In [0]:
def add_metadata(df): # functie:voeg metadata toe aan het DataFrame
    return df.withColumn("ingested_at", current_timestamp())

In [0]:
# api_data = get_api_data(API_CONFIG)

# key_value_data = json_to_df(api_data)

# key_value_data = add_metadata(key_value_data)

# display(api_data) 

# df = json_to_df(api_data)

# display(df)

load()